# 3 · Make a dataset

The 300 tiles we trained on were built from two things: labels experts drew, and the Sentinel-2 picture each label was drawn on. This notebook builds such a dataset ourselves, for three labels, so the same steps work for labels of our own.

It ends with a folder in the layout notebooks 0 to 2 read, plus a catalog that lists every file in it. This notebook needs an internet connection.

In [ ]:
from pathlib import Path

import pandas as pd
from huggingface_hub import snapshot_download

import geosave as gs

# Three labels make the point; the pictures come from the archive, not this download.
names = [
    "dw_-100.2161543359_20.1295348778-20190130",
    "dw_-100.3401185588_19.4460512277-20190130",
    "dw_-100.9639090682_52.4196407429-20190527",
]
patterns = [f"train/labels/{name}.tif" for name in names]
root = Path(snapshot_download(
    "fatmur/dynamicworld-300",
    repo_type="dataset",
    local_dir="data/dynamic_world",
    allow_patterns=patterns,
))
label_paths = [root / pattern for pattern in patterns]

label = gs.open_raster(label_paths[0])
anchor = label.gs.anchor
anchor.format("lat {lat:.2f}, lon {lon:.2f}, {start:%d %B %Y}, {width:g} m across at {res:g} m")

## Where a label's picture comes from

A label knows where its pixels lie and when it was drawn; that pair is its **anchor**. Public archives list pictures by place and date in catalogs that follow the [STAC](https://stacspec.org) standard, so the anchor is enough to search one. Microsoft's Planetary Computer publishes every Sentinel-2 picture.

The archive stores reflectance as whole numbers and does not say which value marks no measurement, so we state the stored type and that value when we set the source up.

In [ ]:
client = gs.stac.StacClient.open("planetary_computer")
bands = ["B02", "B03", "B04", "B08"]
source = client.source("sentinel-2-l2a").set_config(
    bands=[*bands, "SCL"],
    dtype="uint16",
    nodata=0,
)

source.search(anchor)

## Which date

The label's own day might be cloudy over part of the tile, and Sentinel-2 passes every few days. So we search the label's whole month, keep pictures the archive rates at most 20% cloudy, and take the one with the most clear pixels.

`SCL`, the scene classification band, marks each pixel as clear, cloud, shadow and so on; `scl_valid_mask` keeps the clear ones. The archive stores reflectance times 10,000, which `write_packing` records on the picture so it can be decoded later, and `write_rgb` names the bands a colour picture draws, as the shipped tiles do.

In [ ]:
source = source.set_query(filter="eo:cloud_cover <= 20")


def clearest_image(anchor):
    month = anchor.format("{start:%Y-%m}")
    images = source.load(anchor.replace(timespan=month))

    clear = gs.features.scl_valid_mask(images, scl="SCL")
    clear_share = clear.mean(dim=["y", "x"]).compute()
    best = int(clear_share.argmax())

    image = images[bands].isel(time=[best])
    image = image.gs.write_packing(1e-4, target=bands)
    image = image.gs.write_rgb(red="B04", green="B03", blue="B02")
    return image, clear_share


image, clear_share = clearest_image(anchor)
clear_share.to_series()

## Every label, saved

Now the same for every label, each saved beside its picture in the layout notebook 0 reads: one folder per split, `images/` and `labels/` inside, matching file names. With three labels, each split gets one.

`to_cog` returns the paths it wrote. A picture loaded from the archive carries the archive's collection, `sentinel-2-l2a`, and its saved file keeps it.

In [ ]:
dataset = Path("data/my_dataset")


def save_sample(image, label, name, split):
    (image_path,) = image.gs.to_cog(dataset / split / "images" / f"{name}.tif", overwrite=True)
    (label_path,) = label.gs.to_cog(dataset / split / "labels" / f"{name}.tif", overwrite=True)
    return image_path, label_path


image_paths = []
label_files = []
split_of = {}
for path, split in zip(label_paths, ["train", "val", "test"]):
    label = gs.open_raster(path)
    image, _ = clearest_image(label.gs.anchor)

    image_path, label_path = save_sample(image, label, path.stem, split)
    image_paths.append(image_path)
    label_files.append(label_path)
    split_of[path.stem] = split

image_paths

## A catalog of the dataset

A folder of files says nothing about where or when each tile is. A catalog does: one **Item** per tile, with its footprint, its date and its files, called assets. A **Collection** groups Items that share a description; [PySTAC](https://pystac.readthedocs.io) has a class for each.

`create_item` reads each saved file's header, no pixels. The pictures state their own collection; the labels state none, so we name theirs. The catalog is a GeoDataFrame, one row per Item, so the split is one more column. Saved as GeoParquet, it is the file notebook 4 starts from.

In [ ]:
images = gs.stac.create_item(image_paths)
labels = gs.stac.create_item(label_files, collection="label")

catalog = pd.concat([images, labels], ignore_index=True)
catalog["split"] = catalog["id"].map(split_of)

catalog_path = dataset / "catalog.parquet"
catalog.gs.to_geoparquet(catalog_path, overwrite=True)
catalog

## Is it model input?

Read one pair back the way a training loop would. `make_features` and `preprocess` are notebook 2's: reflectance plus two indices on the raster, then a standardized tensor. With one training tile, its own mean and spread stand in for the training set's.

The model is notebook 2's too, untrained here: this checks that a tile we made goes in and a prediction comes back on the label's grid, not that the prediction is any good.

In [ ]:
import torch
from torch import nn

image = gs.open_raster(dataset / "train/images" / f"{label_paths[0].stem}.tif").isel(time=0)
label = gs.open_raster(dataset / "train/labels" / f"{label_paths[0].stem}.tif")["label"].isel(time=0)


def make_features(image):
    reflectance = image.gs.mask_and_scale()
    ndvi = gs.features.ndvi(reflectance, nir="B08", red="B04")
    ndwi = gs.features.ndwi(reflectance, green="B03", nir="B08")
    return reflectance.assign(ndvi=ndvi, ndwi=ndwi)


features = make_features(image)
tensor = features.gs.to_tensor(dtype="float32").nan_to_num(0)
mean = tensor.mean(dim=(1, 2)).view(-1, 1, 1)
std = tensor.std(dim=(1, 2)).view(-1, 1, 1)


def preprocess(features):
    image = features.gs.to_tensor(dtype="float32")
    image = image.nan_to_num(0)
    return (image - mean) / std


model = nn.Sequential(
    nn.Conv2d(6, 16, 3, stride=2, padding=1), nn.ReLU(),
    nn.Upsample(scale_factor=2), nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(),
    nn.Conv2d(16, 10, 1),
)

batch = preprocess(features)[None]  # (1, 6, H, W)
with torch.no_grad():
    logits = model(batch)  # (1, classes, H, W)
prediction = logits.argmax(dim=1)[0]  # (H, W)

codes = prediction.numpy().astype("uint8")
predicted = gs.raster({"label": (("y", "x"), codes)}, label)
label.gs.plot() + predicted.gs.plot()

## Labels drawn as polygons

Labels often arrive as shapes rather than pixels: a field boundary, a forest outline. `data/khdtk_wanadipa_batas.geojson` outlines a forest research area in Central Java. Saying it is all trees, it becomes one more label.

A shape has no grid, so we choose one: `GeoAnchor.from_geometry` covers the shape at 10 m, the pictures' resolution, for July 2024. `rasterize` burns the trees class, 1, into that grid, and 255, unlabelled, everywhere else. Copying a Dynamic World label's attributes gives the new label the same class names and colours. From there it is the same as any label: the clearest picture of its month, saved into the dataset and added to the catalog.

In [ ]:
region = gs.open_vector("data/khdtk_wanadipa_batas.geojson")
region["label"] = 1  # trees

region_anchor = gs.GeoAnchor.from_geometry(region.geometry.iloc[0], resolution=10, timespan="2024-07")
burned = region.gs.rasterize(region_anchor, column="label", fill=255, dtype="uint8")

classes = gs.open_raster(label_paths[0])["label"].attrs
region_label = burned.assign_attrs(classes).to_dataset(name="label")
region_label = region_label.expand_dims(time=[pd.Timestamp("2024-07-01")])

region_image, _ = clearest_image(region_label.gs.anchor)
image_path, label_path = save_sample(region_image, region_label, "khdtk_wanadipa", "train")

new_rows = pd.concat(
    [gs.stac.create_item([image_path]), gs.stac.create_item([label_path], collection="label")],
    ignore_index=True,
)
new_rows["split"] = "train"
catalog = pd.concat([catalog, new_rows], ignore_index=True)
catalog.gs.to_geoparquet(catalog_path, overwrite=True)

gs.stack({"image": region_image.isel(time=0), "label": region_label.isel(time=0)}).gs.plot(cols=2)

## Share it

GeoParquet suits a Python project. People using STAC Browser or QGIS expect STAC's JSON layout instead: one file for the catalog, one per Collection, one per Item. `to_stac` writes it beside the dataset; each Item file holds what this one shows.

In [ ]:
catalog.gs.to_stac(dataset / "stac", overwrite=True)

item = catalog.iloc[[0]].gs.to_item()[0]
item

## What happened

We took labels, found each one's picture in a public archive by its place and date, chose the clearest date of its month, and saved picture and label side by side in the layout notebook 0 reads. A catalog lists every file with its footprint, date and split, saved as GeoParquet for the project and as STAC JSON for sharing. A label drawn as a polygon joined the same dataset once it had a grid.

| Next | Question |
| --- | --- |
| 4 · Fitting data to a model | A catalog of tiles of different sizes is not a model input. How do I cut it into what a model reads? |